# Spatial drought analysis

In [ ]:
# general
import geopandas as gpd
import pandas as pd

# plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from src.utils import load_config, weighted_mean_by_lat

cfg = load_config()

scenarios       = [s["id"] for s in cfg["scenarios"]]
scenarios_names = [s["name"] for s in cfg["scenarios"]]
colors          = [s["color"] for s in cfg["scenarios"]]

## Data

In [ ]:
cities = {
    "Huaraz":            -9.53,
    "Cusco":            -13.52,
    "La Paz":           -16.50,
    "Calama":           -22.46,
    "Vallenar":         -28.50,
    "Mendoza":          -32.70,
    "Santiago":         -33.60,
    "Talca":            -35.43,
    "Temuco":           -38.74,
    "Bariloche":        -41.13,
    "Coyhaique":        -45.57,
    "El Chaltén":       -49.33,
    "El Calafate":      -50.34,
    "Puerto Natales":   -51.73,
    "Tierra del Fuego": -54.80,
}

In [ ]:
# drought metrics
df_metrics = pd.read_csv("data/hydro/droughts_hydroatlas_whatif.csv", index_col=0)
df_metrics = df_metrics.pivot(columns="scenario")
df_metrics.columns = [f"{col[0]}_{col[1]}" for col in df_metrics.columns]

# hydroATLAS + metrics
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")
hydroatlas_shape['basin_cenlat_rounded'] = hydroatlas_shape['basin_cenlat'].round(0)
hydroatlas_shape.gauge_id = hydroatlas_shape.gauge_id.astype(int)
hydroatlas_shape = hydroatlas_shape.set_index("gauge_id")
hydroatlas_shape = hydroatlas_shape.join(df_metrics)
 

# hydroATLAS + drought extent
df_extent = pd.read_csv("data/hydro/droughts_hydroatlas_area_ts.csv", index_col=["scenario", "date"], parse_dates=["date"])
df_extent = df_extent.rolling(window=90, min_periods=1, center=True).mean()
df_extent = df_extent.groupby(level='scenario').mean()
df_extent = df_extent.transpose()
df_extent.index = df_extent.index.astype(int)

for scenario in scenarios:
    df_extent[scenario] = ((df_extent[scenario] / df_extent["historical"]) -1)*100

In [ ]:
# hydroATLAS + water stress
df = pd.read_parquet("data/hydro/water_stress.parquet")
df = df.set_index(["scenario", "month", "gauge_id"])[["WSI"]]

# WSI 
wsi = df[["WSI"]].loc[df.index.get_level_values("month").isin([1, 2, 3])]
wsi = wsi.groupby(["scenario", "gauge_id"]).mean()
wsi = wsi.unstack("scenario").droplevel(0, axis=1)
wsi.columns = "WSI_" + wsi.columns
wsi.index = wsi.index.astype(int)

wsi_historical = wsi["WSI_historical"]
wsi = wsi.div(wsi_historical, axis=0).sub(1).mul(100)

hydroatlas_shape = hydroatlas_shape.join(wsi)
hydroatlas_shape_lat = weighted_mean_by_lat(hydroatlas_shape)

## Plot

In [ ]:
fig = make_subplots(rows=1, cols=4, horizontal_spacing=0.02, shared_yaxes=True)
slides = False

# Panel 1: Min 30-day
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["min_30day_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False), 
                  row=1, col=1)
fig.update_xaxes(title_text="Δ Min 30-day flow (%)", title_standoff=5, title_font_size=15, tickfont_size=13, row=1, col=1)

# Panel 2: Severity
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["severity_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h', name = scenarios_names[scenarios.index(scenario)],
                         showlegend=True), 
                  row=1, col=2)
fig.update_xaxes(title_text="Δ Severity (%)", title_standoff=5, title_font_size=15, tickfont_size=13, row=1, col=2)

# Panel 3: Extent
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=df_extent[scenario], y=df_extent.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False), 
                  row=1, col=3)
fig.update_xaxes(title_text="Δ Extent (%)", title_standoff=5, title_font_size=15, tickfont_size=13, row=1, col=3)

# Panel 4: WSI
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["WSI_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False),  
                  row=1, col=4)
fig.update_xaxes(title_text="Δ WSI summer (%)", title_standoff=5, title_font_size=14, tickfont_size=13, row=1, col=4)


# Add panel labels (a, b, c, d) to lower left of each plot
labels = ['a)', 'b)', 'c)', 'd)']
x_positions = [0.21, 0.46, 0.72, 0.97]  # Adjusted for 4 panels with horizontal_spacing=0.02
for i, (label, x_pos) in enumerate(zip(labels, x_positions)):
    fig.add_annotation(
        text=label,
        x=x_pos,
        y=0.995,
        xref='paper',
        yref='paper',
        showarrow=False,
        font=dict(size=17, family='Arial', color='black'),
        xanchor='left',
        yanchor='top'
    )

# Update settings for each subplot
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False, tickfont_size=13)
fig.update_yaxes(ticks="outside", griddash="dot", range=[-56, -7], ticksuffix="ºS", zeroline=False, tickfont_size=13)

# Add vertical zeroline for each subplot
for col in range(1, 5):
    fig.add_shape(
        type="line",
        x0=0, x1=0,
        y0=-56, y1=-7,
        xref=f"x{col}", yref=f"y{col}",
        line=dict(color="#26293F", width=0.7)
    )


# Add vertical line on the left
fig.add_shape(
    type="line",
    x0=-0.09, x1=-0.09,
    y0=0, y1=1.0,
    xref="paper", yref="paper",
    line=dict(width=1)
)

# Add horizontal lines at region boundaries
for boundary_lat in [-7, -20, -30, -35, -45, -52, -56]:
    y_pos = (boundary_lat - (-56)) / ((-7) - (-56))
    fig.add_shape(
        type="line",
        x0=-0.10, x1=-0.08,
        y0=y_pos, y1=y_pos,
        xref="paper", yref="paper",
        line=dict(width=1)
    )

# Add region annotations on the vertical line
for region, (lat_min, lat_max) in cfg["regions"].items():
    # Calculate midpoint of the region in latitude

    mid_lat = (lat_min + lat_max) / 2
    y_pos = (mid_lat - (-56)) / ((-7) - (-56))
    
    fig.add_annotation(
        text=region.replace(' ', '<br>'),
        bgcolor="white", 
        x=-0.09,
        y=y_pos,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=14, family="Arial"),
        xanchor="center",
        yanchor="middle",
    )

# Add city annotations
for city, lat in cities.items():
    y_pos = (lat - (-56)) / ((-7) - (-56))
    fig.add_annotation(
        text="- " + city,
        x=0,
        y=y_pos,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=13, family="Arial"),
        xanchor="left",
    )

# Figure size and output suffix
width, height, suffix = (1000, 600, "_slides") if slides else (900, 750, "")

fig.update_layout(
    width=width,
    height=height,
    barmode="overlay",
    legend=dict(
        title="Scenarios:",
        title_font_size=15,
        font_size=14,
        orientation="h",
        yanchor="bottom",
        y=1,
        xanchor="right",
        x=1,
    ),
    margin=dict(l=110, r=10, b=10, t=0),
    font=dict(family="Arial", size=15),
)

fig.write_image(
    f"figures/fig_whatif_droughts_latitudinal_v2{suffix}.svg",
    width=width,
    height=height,
)
fig.show()